<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 5 · Manipulación de datos: agrupar, limpiar y combinar

La semana pasada dejaste `ventas.csv` en condiciones y demostraste con una cifra de control que no
perdiste nada. Ese trabajo ya está hecho: desde hoy partes de `ventas_limpias.csv` y el laboratorio
puede dedicarse entero a las dos operaciones que convierten tablas sueltas en un indicador de
negocio. Agrupar contesta preguntas; combinar trae el contexto que las hace interpretables. Y las dos
juntas son el lugar donde nacen los errores más caros del curso: hoy vas a inflar la facturación de
Comercial Andina en 701 246,52 dólares con una sola línea de `merge`, y a medir exactamente cuánto.

> **Hoy haces** · Filtras y derivas con `query` y `assign`, agrupas con `groupby` y `agg` por una y
> por varias claves (90 min). Recorres los cuatro tipos de unión sobre las tablas de Comercial Andina,
> compruebas la cardinalidad antes de aceptarlas y mides el daño de una unión mal hecha. Reestructuras
> con `pivot_table` y cierras construyendo la tabla mensual de indicadores desde cuatro fuentes,
> cuadrada contra una cifra de control externa.
>
> **Entrega** · Este cuaderno ejecutado, la tabla mensual de indicadores con sus 154 filas, las tres
> comprobaciones de toda unión (filas antes, filas después, suma de control) y la explicación escrita
> de la diferencia contra la cifra de control.
> Nombre de archivo: `lab_05_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
REPO = "https://github.com/mayait/CursoAnalisisDatos_IA_2026.git"
COPIA = Path("/content/CursoAnalisisDatos_IA_2026")
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              COPIA / "sitio" / "datos"]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    # En Colab el cuaderno llega solo: se trae el repositorio una sola vez.
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(COPIA)], check=True)
    DATOS = COPIA / "sitio" / "datos"

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Cinco tablas, cinco granularidades

Antes de unir nada hay que poder decir, en una frase, **qué representa una fila** de cada tabla. Es la
pregunta de la semana pasada aplicada a cinco archivos a la vez, y es la que decide qué unión es
correcta: una tabla de hechos se une a una tabla de dimensiones, nunca al revés y nunca entre dos
tablas de hechos.

In [ ]:
ventas = pd.read_csv(DATOS / "ventas_limpias.csv", parse_dates=["fecha"])
clientes = pd.read_csv(DATOS / "clientes.csv")
productos = pd.read_csv(DATOS / "productos.csv")
sucursales = pd.read_csv(DATOS / "sucursales.csv")
marketing = pd.read_csv(DATOS / "marketing_mensual.csv", parse_dates=["mes"])

# La limpieza de la semana 4 sobre las ciudades, en una línea. Sin esto, cualquier
# groupby por ciudad devuelve 25 categorías en lugar de 5.
print(f"ciudades distintas en clientes.csv antes de normalizar : {clientes['ciudad'].nunique()}")
clientes["ciudad"] = (clientes["ciudad"].str.strip().str.title()
                      .replace({"Guayaquíl": "Guayaquil"}))
print(f"ciudades distintas después                             : {clientes['ciudad'].nunique()}"
      f" → {sorted(clientes['ciudad'].unique())}\n")

mapa = pd.DataFrame([
    ("ventas",     len(ventas),     "una línea de factura", "factura_id + producto_id", "hechos"),
    ("clientes",   len(clientes),   "un cliente",           "cliente_id",               "dimensión"),
    ("productos",  len(productos),  "un producto",          "producto_id",              "dimensión"),
    ("sucursales", len(sucursales), "un punto de venta",    "sucursal_id",              "dimensión"),
    ("marketing",  len(marketing),  "un mes",               "mes",                      "hechos"),
], columns=["tabla", "filas", "qué es una fila", "clave", "tipo"])

# Una dimensión sirve para unir solo si su clave es única. Se comprueba, no se supone.
mapa["clave única"] = [ventas.duplicated(["factura_id", "producto_id"]).sum() == 0,
                       clientes["cliente_id"].is_unique, productos["producto_id"].is_unique,
                       sucursales["sucursal_id"].is_unique, marketing["mes"].is_unique]
print(mapa.to_string(index=False))

📌 `ventas` es la tabla de hechos: 79 482 filas, una por línea de factura. Las tres dimensiones tienen
clave única —1 800 clientes, 74 productos, 6 sucursales— y por eso se les puede pegar a las ventas sin
riesgo. `ventas` es la única cuya clave natural no es una sola columna, y `marketing` es la segunda
tabla de hechos: está a nivel de mes, así que **nunca** se une línea a línea con las ventas.

## 2. Filtrar y derivar: `query` y `assign`

`query` filtra escribiendo la condición como se dice en voz alta. `assign` crea columnas calculadas
sin tocar el original y encadenables una detrás de otra. Las dos juntas evitan el patrón que rompe
cuadernos: modificar un DataFrame en su sitio y perder la pista de en qué estado quedó.

In [ ]:
ventas = ventas.assign(
    monto=lambda d: d["cantidad"] * d["precio_unitario"] * (1 - d["descuento"]),
    mes=lambda d: d["fecha"].dt.to_period("M").dt.to_timestamp(),
)
TOTAL_NETO = ventas["monto"].sum()

# query lee como una frase. Ojo: las devoluciones tienen cantidad negativa.
solo_ventas = ventas.query("es_devolucion == False")
devoluciones = ventas.query("es_devolucion == True")
grandes = ventas.query("monto > 200 and descuento > 0")

print(f"líneas totales        : {len(ventas):>8,}   monto {TOTAL_NETO:>14,.2f}  ← facturación NETA")
print(f"  ventas              : {len(solo_ventas):>8,}   monto {solo_ventas['monto'].sum():>14,.2f}")
print(f"  devoluciones        : {len(devoluciones):>8,}   monto {devoluciones['monto'].sum():>14,.2f}")
print(f"  líneas de más de 200 con descuento: {len(grandes):,}\n")

print("La devolución NO es un caso raro: se lleva el "
      f"{-devoluciones['monto'].sum() / solo_ventas['monto'].sum():.2%} de lo facturado.")
print("Decide desde ahora qué total reportas. En este cuaderno, siempre el NETO.")

⚠️ Aquí ya hay dos totales distintos del mismo negocio: 2 874 635,43 si sumas solo las ventas y
2 806 650,55 si restas las devoluciones. La diferencia es el 2,36 %, y es exactamente la razón por la
que dos áreas de la misma empresa reportan cifras que no coinciden. No hay una correcta en abstracto:
hay una **declarada**. En este cuaderno se reporta siempre el neto.

## 3. Dividir, aplicar y recombinar

`groupby` parte la tabla, aplica una función a cada trozo y vuelve a pegar el resultado. Con `agg` y
nombres explícitos —lo que pandas llama *named aggregation*— el resultado sale con las columnas ya
bautizadas, listo para un informe y sin índices de dos pisos.

In [ ]:
por_sucursal = (ventas
    .groupby("sucursal_id")
    .agg(facturacion=("monto", "sum"),
         lineas=("monto", "size"),
         facturas=("factura_id", "nunique"),
         clientes=("cliente_id", "nunique"),
         unidades=("cantidad", "sum"))
    .sort_values("facturacion", ascending=False))

por_sucursal["ticket_medio"] = por_sucursal["facturacion"] / por_sucursal["facturas"]
por_sucursal["% del total"] = por_sucursal["facturacion"] / TOTAL_NETO * 100
por_sucursal.round(2)

Cinco líneas de código y ya hay una lectura de negocio: **S99, el canal en línea, es el tercero en
facturación con 510 172,84 pero el primero en clientes distintos con 1 429**, más del doble que
cualquier tienda física. Vende poco a mucha gente; su ticket medio de 103,82 es el más bajo de la red y
la mitad del de Cuenca, que factura 203,91 por visita. Es una tienda con otro modelo de negocio metida en la misma tabla.

Con varias claves el `groupby` abre la pregunta en dos dimensiones. Y ahí es donde el `merge` empieza
a hacer falta: `ventas` no sabe en qué ciudad está el cliente ni de qué tipo es.

In [ ]:
enriquecidas = ventas.merge(clientes[["cliente_id", "ciudad", "tipo_cliente"]],
                            on="cliente_id", how="left", validate="m:1")

perfil = (enriquecidas
    .groupby(["ciudad", "tipo_cliente"])
    .agg(facturacion=("monto", "sum"),
         clientes=("cliente_id", "nunique"),
         facturas=("factura_id", "nunique"))
    .assign(ticket=lambda d: d["facturacion"] / d["facturas"],
            pct=lambda d: d["facturacion"] / d["facturacion"].sum() * 100))

resumen_tipo = perfil.groupby("tipo_cliente")[["facturacion", "clientes"]].sum()
resumen_tipo["% facturación"] = resumen_tipo["facturacion"] / TOTAL_NETO * 100
resumen_tipo["% clientes"] = resumen_tipo["clientes"] / resumen_tipo["clientes"].sum() * 100
print(resumen_tipo.round(2).to_string(), "\n")

perfil.round(2)

📌 El 21,82 % de los clientes que compran son mayoristas y se llevan el **92,05 %** de la facturación.
El ticket mayorista ronda los 400 en las cinco ciudades y el minorista los 20: son dos negocios, no
dos segmentos. Cualquier promedio calculado sobre los dos juntos no describe a ningún cliente real. Es la trampa del promedio de la semana 3,
reapareciendo ahora en una tabla agrupada.

La suma no cambia nunca: 2 806 650,55 en los cuatro niveles. Lo que cambia es el promedio, que va de
35,31 por línea a 1 602,88 por cliente. Y fíjate en la última columna: la razón media/mediana empeora
al subir de nivel —de 3,13 en la línea a 9,26 en el cliente— porque agregar concentra a los mayoristas.
**Si alguien te dice «el ticket promedio es X», la primera pregunta es de qué tabla salió.**

## 4. Los cuatro tipos de unión

La teoría cabe en una frase: la unión decide **qué filas sobreviven cuando la clave no está en las dos
tablas**. Lo que la hace difícil no es la sintaxis sino que las cuatro variantes corren sin error y
devuelven totales distintos. Comercial Andina tiene el caso perfecto para verlo: hay clientes en el
padrón que nunca compraron.

In [ ]:
resumen_cliente = (ventas.groupby("cliente_id")
                   .agg(facturacion=("monto", "sum"),
                        facturas=("factura_id", "nunique"),
                        ultima_compra=("fecha", "max"))
                   .reset_index())

filas = []
for tipo, caso in [("inner", "clientes que compraron: la intersección"),
                   ("left", "todo lo vendido, con los datos del cliente al lado"),
                   ("right", "el padrón completo, incluidos los que nunca compraron"),
                   ("outer", "todo de las dos tablas")]:
    j = resumen_cliente.merge(clientes, on="cliente_id", how=tipo)
    filas.append((tipo, len(j), int(j["facturacion"].isna().sum()),
                  j["facturacion"].sum(), caso))

print(f"resumen_cliente: {len(resumen_cliente):,} filas   ·   clientes.csv: {len(clientes):,} filas\n")
comparacion = pd.DataFrame(filas, columns=["how", "filas", "facturación nula",
                                           "facturación total", "para qué sirve"])
print(comparacion.to_string(index=False))

Las cuatro uniones devuelven **la misma facturación** —2 806 650,55— porque ninguna duplica filas.
Lo único que cambia es a cuántos clientes se la reparten. Y ahí está el negocio: el `right join` saca a
la luz **49 clientes del padrón que nunca compraron nada**, el 2,7 % de la base captada. El `inner
join`, que es el que casi todo el mundo escribe por costumbre, los borra sin decir nada.

- **Interna** · para calcular una tasa de conversión sobre los que sí compraron.
- **Izquierda** · para enriquecer la tabla de hechos sin perder ni una venta. Es la que más se usa.
- **Derecha** · para auditar la dimensión: ¿a quién captamos y nunca le vendimos?
- **Externa** · para conciliar dos sistemas y ver qué falta en cada uno.

In [ ]:
sin_compras = clientes[~clientes["cliente_id"].isin(ventas["cliente_id"])]

print(f"clientes captados que nunca compraron: {len(sin_compras)} "
      f"({len(sin_compras) / len(clientes):.1%} del padrón)\n")
print(pd.crosstab(sin_compras["canal_captacion"], sin_compras["tipo_cliente"],
                  margins=True, margins_name="Total").to_string())
print("\nTasa de activación por canal de captación:")
activacion = pd.DataFrame({
    "captados": clientes["canal_captacion"].value_counts(),
    "compraron": clientes[clientes["cliente_id"].isin(ventas["cliente_id"])]["canal_captacion"].value_counts(),
})
activacion["% activados"] = (activacion["compraron"] / activacion["captados"] * 100).round(1)
print(activacion.sort_values("% activados").to_string())

Las cuatro tasas de activación caben en ocho décimas: **96,80 % en Referido y 97,60 % en Punto de
venta y en Redes sociales**. La conclusión honesta es negativa —ningún canal capta clientes que luego
no compran, los 49 son ruido repartido entre los cuatro— y es exactamente el tipo de conclusión que
hay que saber escribir. Lo importante no es el hallazgo, es que **la pregunta solo se puede formular
con el `right join`**: con una unión interna esos 49 clientes no existen y el número no se puede
calcular.

## 5. Cardinalidad: la comprobación que se hace antes, no después

Cardinalidad es cuántas filas de una tabla corresponden a cada fila de la otra. Uno a uno y muchos a
uno son seguras. **Muchos a muchos multiplica filas**, y pandas la ejecuta sin una sola advertencia.
El argumento `validate` convierte tu supuesto en una comprobación que revienta si es falso, que es
justo lo que quieres que pase.

In [ ]:
# Las tres uniones de la tabla de hechos con sus dimensiones, declarando la cardinalidad.
ventas_completas = (ventas
    .merge(clientes[["cliente_id", "ciudad", "tipo_cliente", "canal_captacion"]],
           on="cliente_id", how="left", validate="m:1")
    .merge(productos[["producto_id", "categoria", "subcategoria", "costo_unitario"]],
           on="producto_id", how="left", validate="m:1")
    .merge(sucursales[["sucursal_id", "canal", "metros_cuadrados"]],
           on="sucursal_id", how="left", validate="m:1"))

ventas_completas["costo"] = ventas_completas["cantidad"] * ventas_completas["costo_unitario"]
ventas_completas["margen"] = ventas_completas["monto"] - ventas_completas["costo"]

print(f"filas antes  : {len(ventas):,}")
print(f"filas después: {len(ventas_completas):,}   ← idénticas, como tiene que ser en m:1")
print(f"suma de control: {ventas['monto'].sum():,.2f} → {ventas_completas['monto'].sum():,.2f}")
print(f"huérfanos (filas sin pareja en alguna dimensión): "
      f"{int(ventas_completas[['ciudad', 'categoria', 'canal']].isna().any(axis=1).sum())}\n")

# Qué pasa si el supuesto es falso: validate lo dice en voz alta en lugar de multiplicar en silencio.
try:
    ventas.merge(clientes, on="cliente_id", how="left", validate="1:1")
except Exception as e:
    print(f"validate='1:1' sobre una unión muchos-a-uno →\n  {type(e).__name__}: {e}")

In [ ]:
# ✅ Comprobación 1 · las tres comprobaciones obligatorias de toda unión
assert len(ventas_completas) == len(ventas), \
    f"La unión cambió el número de filas ({len(ventas):,} → {len(ventas_completas):,}): revisa la cardinalidad"
assert abs(ventas_completas["monto"].sum() - ventas["monto"].sum()) < 0.01, \
    "La suma de control se movió con la unión: alguna dimensión tiene claves repetidas"
assert ventas_completas["ciudad"].isna().sum() == 0, \
    "Hay filas huérfanas sin ciudad: algún cliente_id de ventas no existe en clientes.csv"
assert abs(ventas_completas["monto"].sum() - 2806650.55) < 1.0, \
    "Tu facturación no coincide con la cifra de control del curso (2 806 650,55)"

print(f"Comprobación 1 superada ✓  {len(ventas_completas):,} filas intactas y "
      f"{ventas_completas['monto'].sum():,.2f} de facturación")

Filas idénticas antes y después, cero huérfanos y la suma de control intacta. Esas son las **tres
comprobaciones obligatorias de toda unión** de aquí en adelante, y las tres caben en tres líneas de
código. El `validate` falla a propósito en la última prueba: no es un error del cuaderno, es la
herramienta haciendo su trabajo.

Con la tabla completa ya se puede calcular algo que ninguna de las cinco fuentes contiene por separado:
el margen.

In [ ]:
margen_categoria = (ventas_completas
    .groupby("categoria")
    .agg(facturacion=("monto", "sum"), costo=("costo", "sum"), margen=("margen", "sum"))
    .assign(margen_pct=lambda d: d["margen"] / d["facturacion"] * 100)
    .sort_values("margen_pct"))

print(f"margen global de Comercial Andina: {ventas_completas['margen'].sum():,.2f} "
      f"({ventas_completas['margen'].sum() / TOTAL_NETO:.2%} sobre ventas)\n")
margen_categoria.round(2)

📌 Abarrotes factura 640 119,90 y deja 38,01 % de margen; Bebidas factura la mitad, 321 584,90, y deja
53,96 %. Ordenar por facturación y ordenar por margen dan dos rankings distintos, y solo el segundo
sirve para decidir qué se promociona. Este número no existía en ninguna tabla: nace de unir ventas con
el costo del catálogo.

## 6. La tabla mensual de indicadores, desde las cuatro fuentes

Este es el entregable de la semana y el patrón que vas a repetir todo el semestre: **hechos + tres
dimensiones → una fila por mes y ciudad → cuadre contra una cifra externa**. Cada indicador declara de
qué fuente viene, porque en la revisión siempre preguntan eso.

In [ ]:
indicadores = (ventas_completas
    .groupby(["mes", "ciudad"])
    .agg(facturacion=("monto", "sum"),          # ventas
         margen=("margen", "sum"),              # ventas + productos
         unidades=("cantidad", "sum"),          # ventas
         facturas=("factura_id", "nunique"),    # ventas
         clientes_activos=("cliente_id", "nunique"))  # ventas + clientes (la ciudad)
    .reset_index())

indicadores["ticket_medio"] = indicadores["facturacion"] / indicadores["facturas"]
indicadores["margen_pct"] = indicadores["margen"] / indicadores["facturacion"] * 100

# La cuarta fuente: los metros cuadrados de la tienda física de cada ciudad.
m2 = sucursales.query("canal == 'Tienda'").set_index("ciudad")["metros_cuadrados"]
en_tienda = (ventas_completas.query("canal == 'Tienda'")
             .groupby(["mes", "ciudad"])["monto"].sum().rename("facturacion_tienda").reset_index())
indicadores = indicadores.merge(en_tienda, on=["mes", "ciudad"], how="left", validate="1:1")
indicadores["m2"] = indicadores["ciudad"].map(m2)
indicadores["venta_por_m2"] = indicadores["facturacion_tienda"] / indicadores["m2"]

print(f"tabla de indicadores: {indicadores.shape[0]} filas × {indicadores.shape[1]} columnas")
print(f"meses {indicadores['mes'].nunique()} × ciudades {indicadores['ciudad'].nunique()} = "
      f"{indicadores['mes'].nunique() * indicadores['ciudad'].nunique()} combinaciones posibles\n")
indicadores.head(5).round(2)

154 filas y no 155: en julio de 2026 Loja no tuvo ni una sola línea, ni venta ni devolución. `groupby`
no inventa la fila, y eso está bien: **una fila ausente y un cero no son lo mismo**, igual que la
semana pasada un nulo no era un cero.

Ahora el cuadre. La cifra de control es `marketing_mensual.csv`, que llega del área de marketing con la
facturación mensual que ellos reportan. Es una fuente independiente de la que acabas de construir, y
por eso sirve para validar.

In [ ]:
control = (indicadores.groupby("mes")["facturacion"].sum().rename("mi_tabla").reset_index()
           .merge(marketing[["mes", "ventas_mes"]].rename(columns={"ventas_mes": "control"}),
                  on="mes", how="outer", validate="1:1"))
control["diferencia"] = control["mi_tabla"] - control["control"]

suma_mia, suma_control = control["mi_tabla"].sum(), control["control"].sum()
print(f"meses en mi tabla        : {control['mi_tabla'].notna().sum()}")
print(f"meses en la cifra control: {control['control'].notna().sum()}")
print(f"facturación construida   : {suma_mia:,.2f}")
print(f"facturación de control   : {suma_control:,.2f}")
print(f"diferencia total         : {suma_mia - suma_control:,.4f}  "
      f"({abs(suma_mia - suma_control) / suma_control:.8%})")
print(f"peor mes                 : {control['diferencia'].abs().max():.4f}\n")
print(control.head(4).to_string(index=False))

In [ ]:
# ✅ Comprobación 2 · la tabla mensual cuadra contra una fuente independiente
assert len(indicadores) == 154, \
    f"La tabla de indicadores debería tener 154 filas y tiene {len(indicadores)}"
assert abs(suma_mia - suma_control) < 0.01, \
    "Tu facturación mensual no cuadra con marketing_mensual.csv: busca el mes que se desvía"
assert indicadores["facturacion"].isna().sum() == 0, \
    "Hay meses con facturación vacía: revisa el merge con validate='1:1'"

print(f"Comprobación 2 superada ✓  154 filas y una diferencia de "
      f"{suma_mia - suma_control:,.4f} dólares contra la cifra de control")

**Cuadra.** La diferencia total es de 0,0035 dólares sobre 2,8 millones y el peor mes se desvía medio
centavo: es el redondeo a dos decimales de la fuente de control, no un error de construcción. Esa es la
frase que va al informe, y va con el número: *«la tabla reproduce la facturación reportada por
marketing con una diferencia de 0,0035 dólares, atribuible a redondeo»*.

Si la diferencia hubiera sido del 3 %, la respuesta correcta **no** es ajustar hasta que cuadre: es
encontrar de dónde salen esos puntos. Casi siempre son devoluciones contadas de forma distinta, un mes
de corte que no coincide o una unión que duplicó filas. Que es lo que viene ahora.

### 🌶️ Ejercicio 1 — Guiado

Añade tres indicadores más a la tabla `indicadores`, uno por cada fuente que todavía no exportaste
del todo: el **porcentaje de facturación mayorista** del mes y ciudad (viene de `clientes`), las
**unidades devueltas** (viene de `ventas`) y la **categoría que más margen deja** en ese mes y ciudad
(viene de `productos`). Comprueba después que la facturación total no cambió.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: para el % mayorista, agrupa por ["mes", "ciudad", "tipo_cliente"] y usa unstack()
# Pista 2: para las devoluciones, ventas_completas.query("es_devolucion") y luego groupby
# Pista 3: para la categoría top, groupby(["mes","ciudad","categoria"])["margen"].sum()
#          y después .idxmax() sobre el nivel de categoría
# Verificación obligatoria: filas antes, filas después y suma de facturacion antes y después

### 🔥 Desafío

El gerente comercial pregunta: **¿qué clientes compraban en 2024 y dejaron de comprar en 2026?**
Constrúyelo con uniones, no con filtros: una tabla de facturación por cliente en 2024, otra en 2026, y
la unión que hace visible exactamente a los que están en la primera y no en la segunda. Después ponle
precio: cuánto facturaban esos clientes al año.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: hay dos caminos, merge con how="left" e indicator=True, o merge con how="outer".
#          El argumento indicator=True crea una columna _merge con left_only / both / right_only
# Pista 2: cuidado con el año 2026, que solo tiene medio año de datos: ¿comparas años completos
#          o doce meses móviles? Escribe cuál elegiste y por qué
# Pista 3: cruza el resultado con clientes.csv para ver si el abandono se concentra en algún
#          tipo de cliente, ciudad o canal de captación

### 🎯 Reto en clase (15 min)

Torneo de preguntas, en equipos. Cada equipo escribe **tres preguntas de negocio** que se contesten
con `groupby` o con `merge` sobre las cinco tablas —del estilo *«¿qué subcategoría concentra las
devoluciones?»* o *«¿qué ciudad depende más del canal en línea?»*— y se las pasa al equipo de al lado.
Quince minutos para responder las tres con código. Se puntúa la respuesta que trae el número **y** la
frase de qué decisión habilita; una tabla sin frase no puntúa.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: casi todas las preguntas de negocio caben en este molde. Rellena las cuatro ranuras.
#   (ventas_completas
#    .query("<filtro>")
#    .groupby(["<clave 1>", "<clave 2>"])
#    .agg(<indicador>=("<columna>", "<función>"))
#    .sort_values("<indicador>", ascending=False))

## La trampa de hoy

⚠️ **Unir dos tablas sin verificar el número de filas antes y después.** No falla, no avisa y no deja
huella: produce una tabla más grande, con las mismas columnas y todos los totales inflados.

El escenario es el más común de todos. Pides al área comercial la lista de clientes con sus contactos
y te llega el padrón del CRM, que tiene **una fila por contacto**, no una por cliente: los clientes
grandes aparecen dos y tres veces, uno por cada persona registrada. Aquí lo construimos en código para
poder medir el daño exacto.

In [ ]:
# El padrón de contactos tal como lo exporta el CRM: una fila por persona de contacto.
rng = np.random.default_rng(SEED)
n_contactos = rng.choice([1, 2, 3], size=len(clientes), p=[0.82, 0.14, 0.04])
padron = pd.DataFrame([
    dict(cliente_id=cid, contacto=f"Contacto {k + 1} de {razon}",
         rol=["Compras", "Gerencia", "Bodega"][k])
    for cid, razon, n in zip(clientes["cliente_id"], clientes["razon_social"], n_contactos)
    for k in range(n)])

print(f"clientes en el padrón          : {padron['cliente_id'].nunique():,}")
print(f"filas del padrón               : {len(padron):,}")
print(f"clientes con más de un contacto: {(padron['cliente_id'].value_counts() > 1).sum():,}\n")
print(padron.head(4).to_string(index=False))

# La línea que un asistente de IA escribe sin pestañear, y que nadie revisa.
inflado = ventas.merge(padron, on="cliente_id", how="left")

correcto_filas, inflado_filas = len(ventas), len(inflado)
correcto_monto, inflado_monto = ventas["monto"].sum(), inflado["monto"].sum()

print("                          CORRECTO          CON LA UNIÓN MAL HECHA")
print(f"filas                {correcto_filas:>14,}   {inflado_filas:>22,}"
      f"   (×{inflado_filas / correcto_filas:.4f})")
print(f"facturación          {correcto_monto:>14,.2f}   {inflado_monto:>22,.2f}"
      f"   (×{inflado_monto / correcto_monto:.4f})")
print(f"facturas distintas   {ventas['factura_id'].nunique():>14,}   "
      f"{inflado['factura_id'].nunique():>22,}   ← idénticas: por eso nadie lo nota")
print(f"clientes distintos   {ventas['cliente_id'].nunique():>14,}   "
      f"{inflado['cliente_id'].nunique():>22,}   ← idénticos")
print(f"\nfacturación inventada: {inflado_monto - correcto_monto:,.2f} "
      f"({inflado_monto / correcto_monto - 1:+.2%})")

Diecinueve mil filas nuevas y **701 246,50 dólares de facturación que no existen**, un 24,99 % de
inflación. Y mira las dos últimas líneas: el número de facturas y el de clientes **no cambian**. Los
controles que la mayoría de la gente mira —«¿siguen siendo 17 675 facturas? sí»— pasan limpios.

Lo peor no es el total. Es que el error **no reparte el daño por igual**, porque los clientes con
varios contactos no están distribuidos al azar entre las ciudades.

El reparto del daño por ciudad está en el apéndice. Lo que sí entra hoy es la defensa: tres formas de que esto no te pase.

In [ ]:
# Opción A · declarar la cardinalidad y dejar que pandas te frene.
try:
    ventas.merge(padron, on="cliente_id", how="left", validate="m:1")
except Exception as e:
    print(f"A · validate='m:1' → {type(e).__name__}\n")

# Opción B · colapsar la dimensión a una fila por clave ANTES de unir.
padron_unico = (padron.sort_values(["cliente_id", "rol"])
                .drop_duplicates("cliente_id", keep="first"))
bien = ventas.merge(padron_unico, on="cliente_id", how="left", validate="m:1")
print(f"B · padrón colapsado a {len(padron_unico):,} filas → unión de {len(bien):,} filas, "
      f"monto {bien['monto'].sum():,.2f}")

# Opción C · las tres comprobaciones, siempre las mismas, en una función reutilizable.
def unir_verificando(izq, der, on, how="left", control="monto"):
    antes_filas, antes_suma = len(izq), izq[control].sum()
    res = izq.merge(der, on=on, how=how)
    ok = len(res) == antes_filas and abs(res[control].sum() - antes_suma) < 0.01
    print(f"C · filas {antes_filas:,} → {len(res):,} · {control} {antes_suma:,.2f} → "
          f"{res[control].sum():,.2f} · {'OK' if ok else '¡REVISAR!'}")
    return res

_ = unir_verificando(ventas, padron_unico, on="cliente_id")
_ = unir_verificando(ventas, padron, on="cliente_id")

La opción C es la que se queda contigo el resto del semestre. Doce líneas que imprimen filas antes,
filas después y suma de control, y que dicen `¡REVISAR!` en voz alta cuando algo se movió. Un `merge`
que no imprime esas tres cifras no está terminado.

## Entregable

Sube `lab_05_apellido.ipynb` con:

- La tabla mensual de indicadores construida desde las cuatro fuentes, con sus 154 filas y las
  columnas de facturación, margen, ticket medio, clientes activos y venta por metro cuadrado.
- El cuadre contra la cifra de control de marketing, con la diferencia expresada en dólares y en
  porcentaje, y una frase que explique a qué se debe.
- La demostración de la unión mal hecha con los dos números lado a lado: 2 806 650,55 contra
  3 507 897,07, y las tres comprobaciones que lo detectan.
- Los tres ejercicios resueltos, cada uno con sus tres comprobaciones de unión.
- Una fila nueva en la bitácora de prompts: le pediste al asistente una unión entre dos tablas y
  comprobaste filas antes, filas después y suma de la columna de control. Escribe qué salió.

## Para tu equipo

- La tabla de indicadores del negocio del caso es el entregable de esta semana, y tiene que ser
  **reproducible y cuadrada**: se ejecuta de cero y da el mismo número, y ese número coincide con una
  fuente que no construyeron ustedes —un reporte de contabilidad, una factura mensual, un total del
  ERP—. Si no existe esa fuente externa, consíganla antes que el código.
- Escriban al lado de cada indicador de qué tabla sale. En la defensa se pregunta, y no se puede
  contestar mirando el cuaderno.
- El molde de la función `unir_verificando` va al repositorio del grupo hoy mismo. A partir de esta
  semana, cualquier `merge` del proyecto que no imprima las tres cifras se devuelve sin revisar.

## Apéndice · para profundizar fuera de clase

Lo que sigue **no compite por los noventa minutos de clase**: es opcional y está aquí para quien quiera
llegar más lejos, o para consultarlo cuando el proyecto lo pida. Se ejecuta después de haber corrido
todas las celdas anteriores.

### A1 · Lo que se pierde al agregar

Agregar es tirar información a cambio de poder leer el resultado. La pregunta no es cuánta se pierde
sino **qué pregunta contesta cada nivel**, porque el mismo negocio da tres promedios distintos y los
tres son correctos.

In [ ]:
niveles = pd.DataFrame([
    ("línea de factura", len(ventas), ventas["monto"].mean(), ventas["monto"].median(),
     "¿cuánto pesa un producto en el carrito?"),
    ("factura", ventas["factura_id"].nunique(),
     ventas.groupby("factura_id")["monto"].sum().mean(),
     ventas.groupby("factura_id")["monto"].sum().median(),
     "¿cuánto deja una visita? (ticket)"),
    ("cliente", ventas["cliente_id"].nunique(),
     ventas.groupby("cliente_id")["monto"].sum().mean(),
     ventas.groupby("cliente_id")["monto"].sum().median(),
     "¿cuánto vale un cliente en dos años y medio?"),
    ("mes", ventas["mes"].nunique(),
     ventas.groupby("mes")["monto"].sum().mean(),
     ventas.groupby("mes")["monto"].sum().median(),
     "¿cómo va el negocio?"),
], columns=["nivel", "filas", "media", "mediana", "pregunta que contesta"])
niveles["media/mediana"] = (niveles["media"] / niveles["mediana"]).round(2)

print(niveles.to_string(index=False))
print(f"\nLos cuatro niveles suman lo mismo: {TOTAL_NETO:,.2f}")

### A2 · Reestructurar: de formato largo a formato ancho

`groupby` devuelve formato largo —una fila por combinación— que es ideal para seguir calculando y
malísimo para leer. `pivot_table` lo gira: una fila por mes, una columna por ciudad. Es la misma
información con otra forma, y la forma decide si alguien la entiende.

In [ ]:
tabla_ciudad = pd.pivot_table(
    ventas_completas, index="mes", columns="ciudad", values="monto",
    aggfunc="sum", fill_value=0, margins=True, margins_name="Total")

meses = tabla_ciudad.drop(index="Total")["Total"]
print(f"forma de la tabla: {tabla_ciudad.shape[0]} filas × {tabla_ciudad.shape[1]} columnas")
print(f"esquina inferior derecha (el total de todo): {tabla_ciudad.loc['Total', 'Total']:,.2f}")
print(f"mes más alto : {meses.idxmax():%b-%Y} con {meses.max():,.2f}")
print(f"mes más bajo : {meses.idxmin():%b-%Y} con {meses.min():,.2f}\n")
print("Últimos seis meses y el total, en dólares:")
tabla_ciudad.tail(7).round(0)

Dos cosas saltan de la tabla y ninguna se veía en formato largo. **Diciembre de 2025 factura
132 316,88, el mes más alto de los 31**, y **julio de 2026 aparece en negativo, con −776,80**:
no es un error, son notas de crédito de ventas de junio que se registraron después del cierre. Un mes
con facturación negativa es un caso de manual de que la tabla necesita una nota al pie.

`pivot_table` acepta varias agregaciones a la vez, y ahí es donde deja de ser un giro para convertirse
en un informe.

In [ ]:
resumen_canal = pd.pivot_table(
    ventas_completas, index="tipo_cliente", columns="canal",
    values=["monto", "factura_id"],
    aggfunc={"monto": "sum", "factura_id": "nunique"}, fill_value=0)
resumen_canal.columns = [f"{a}·{b}" for a, b in resumen_canal.columns]

print(resumen_canal.round(2).to_string())
print("\nTicket medio por tipo de cliente y canal:")
ticket = (ventas_completas.groupby(["tipo_cliente", "canal"])
          .agg(facturacion=("monto", "sum"), facturas=("factura_id", "nunique"))
          .assign(ticket=lambda d: d["facturacion"] / d["facturas"])["ticket"]
          .unstack().round(2))
print(ticket.to_string())

El ticket casi no cambia de canal —el mayorista deja 423,90 en línea y 406,78 en tienda; el minorista,
19,77 y 19,56—, así que el canal no explica el tamaño de la compra: lo explica el tipo de cliente.
Lo que sí cambia es quién entra: en línea hay 3 892 facturas minoristas contra 1 022 mayoristas, casi
cuatro a uno. Y aun así, de los 510 172,84 que factura el canal en línea, **433 230,72 —el 84,9 %—
los ponen los mayoristas**. El canal parece minorista si cuentas facturas y es mayorista si cuentas
dinero. Cuál de las dos lecturas usas depende de si estás decidiendo servidores o inventario.

### A3 · El daño de la unión duplicada, repartido por ciudad

El total inflado ya lo viste. Lo que no se ve en el total es que el error **no reparte el daño por igual**, porque los clientes con varios contactos no están distribuidos al azar entre las ciudades: un informe por ciudad quedaría mal en unas plazas y bien en otras.

In [ ]:
por_ciudad = pd.DataFrame({
    "correcto": ventas_completas.groupby("ciudad")["monto"].sum(),
    "inflado": (inflado.merge(clientes[["cliente_id", "ciudad"]], on="cliente_id", how="left")
                .groupby("ciudad")["monto"].sum()),
})
por_ciudad["inflación %"] = (por_ciudad["inflado"] / por_ciudad["correcto"] - 1) * 100
por_ciudad["% real"] = por_ciudad["correcto"] / por_ciudad["correcto"].sum() * 100
por_ciudad["% inflado"] = por_ciudad["inflado"] / por_ciudad["inflado"].sum() * 100
por_ciudad["rank real"] = por_ciudad["correcto"].rank(ascending=False).astype(int)
por_ciudad["rank inflado"] = por_ciudad["inflado"].rank(ascending=False).astype(int)
print(por_ciudad.round(2).to_string(), "\n")

fig, ax = plt.subplots(figsize=(10, 4))
orden = por_ciudad.sort_values("correcto", ascending=False)
x = np.arange(len(orden))
ax.bar(x - 0.2, orden["correcto"], 0.4, label="facturación real", color="#4C72B0")
ax.bar(x + 0.2, orden["inflado"], 0.4, label="con la unión mal hecha", color="#C44E52")
for i, (real, malo) in enumerate(zip(orden["correcto"], orden["inflado"])):
    ax.text(i + 0.2, malo, f"+{malo / real - 1:.0%}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x, orden.index)
ax.set_ylabel("facturación acumulada")
ax.set_title("La unión mal hecha no infla parejo: Loja crece 54 % y Guayaquil 21 %")
ax.legend()
plt.tight_layout()
plt.show()

📌 Loja se infla un 54,47 % y Guayaquil un 21,16 %: el daño **no reparte parejo**. Aquí el orden de
las cinco ciudades aguanta de milagro, pero el peso de cada una no: Loja pasa de valer el 7,65 % del
negocio a valer el 9,45 %, y Quito baja del 36,19 % al 35,26 % sin haber vendido un dólar menos. Las
decisiones que dependen de comparar ciudades —dónde abrir, a quién dar presupuesto, a qué gerente
felicitar— se toman con esos porcentajes. Un error que inflara todo por igual sería casi inofensivo
para las comparaciones; este no lo es, y con otro reparto de contactos también se habría llevado el
orden por delante.

El arreglo son tres líneas, y ninguna es el `merge`.